## df = spark.read.format('delta').load(f"schema.tabela")

Gera um dataframe para cada tabela delta de bronze.


In [0]:
df_agent_policies  = spark.read.format("delta").table("bronze.agent_policies")
df_agents          = spark.read.format("delta").table("bronze.agents")
df_claims          = spark.read.format("delta").table("bronze.claims")
df_customers       = spark.read.format("delta").table("bronze.customers")
df_insurance_types = spark.read.format("delta").table("bronze.insurance_types")
df_payments        = spark.read.format("delta").table("bronze.payments")
df_policies        = spark.read.format("delta").table("bronze.policies")

## df = df_apolice.withColumn("nome_coluna", "valor")

Adiciona uma nova coluna (metadado) de data e hora de processamento e nome do arquivo de origem.

In [0]:
from pyspark.sql.functions import current_timestamp, lit

df_agent_policies = df_agent_policies.withColumn("date_hour_silver", current_timestamp()).withColumn("table_name", lit("agent_policies"))
df_agents         = df_agents.withColumn("date_hour_silver", current_timestamp()).withColumn("table_name", lit("agents"))
df_claims         = df_claims.withColumn("date_hour_silver", current_timestamp()).withColumn("table_name", lit("claims"))
df_customers      = df_customers.withColumn("date_hour_silver", current_timestamp()).withColumn("table_name", lit("customers"))
df_insurance_types= df_insurance_types.withColumn("date_hour_silver", current_timestamp()).withColumn("table_name", lit("insurance_types"))
df_payments       = df_payments.withColumn("date_hour_silver", current_timestamp()).withColumn("table_name", lit("payments"))
df_policies       = df_policies.withColumn("date_hour_silver", current_timestamp()).withColumn("table_name", lit("policies"))

## df_apolice.write.format('delta').saveAsTable("nome_tabela")

Salva os dataframes em arquivos delta lake (formato de arquivo) no schema/database "bronze". As tabelas geradas são do tipo MANAGED (gerenciadas).

In [0]:
# df_apolice.write.format('delta').mode("overwrite").saveAsTable("bronze.apolice")
# df_carro.write.format('delta').mode("overwrite").saveAsTable("bronze.carro")
# df_cliente.write.format('delta').mode("overwrite").saveAsTable("bronze.cliente")
# df_endereco.write.format('delta').mode("overwrite").saveAsTable("bronze.endereco")
# df_estado.write.format('delta').mode("overwrite").saveAsTable("bronze.estado")
# df_marca.write.format('delta').mode("overwrite").saveAsTable("bronze.marca")
# df_modelo.write.format('delta').mode("overwrite").saveAsTable("bronze.modelo")
# df_municipio.write.format('delta').mode("overwrite").saveAsTable("bronze.municipio")
# df_regiao.write.format('delta').mode("overwrite").saveAsTable("bronze.regiao")
# df_sinistro.write.format('delta').mode("overwrite").saveAsTable("bronze.sinistro")
# df_telefone.write.format('delta').mode("overwrite").saveAsTable("bronze.telefone")

## Maiusculas, tirando siglas, etc e gravando no formato delta no Silver

Aplicando Data Quality

In [0]:
from pyspark.sql import functions as F

# ---------- Helpers ----------
def _apply_name_rules(colname: str) -> str:
    """Only normalize to uppercase (dataset already clean)."""
    return colname.upper()

def _safe_drop(df, cols):
    """Drop columns only if they exist."""
    existing = set(df.columns)
    to_drop = [c for c in cols if c in existing]
    return df.drop(*to_drop) if to_drop else df

# ---------- Core ----------
def transform_bronze_to_silver(src_fqn: str):
    """
    Reads a Bronze Delta table, applies light normalization,
    removes Bronze metadata, adds Silver metadata, and writes to Silver.
    """

    # Determine destination table
    table_name = src_fqn.split(".")[1]
    dest_fqn = f"silver.{table_name}"

    # Read Bronze table
    df = spark.read.format("delta").table(src_fqn)

    # Rename columns (uppercase)
    new_cols = [_apply_name_rules(c) for c in df.columns]
    df = df.toDF(*new_cols)

    # Drop Bronze metadata
    df = _safe_drop(df, ["date_hour_bronze", "file_name"])

    # Add Silver metadata
    df = (
        df
        .withColumn("date_hour_silver", F.current_timestamp())
        .withColumn("table_name", F.lit(table_name))
    )

    # Write to Silver
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(dest_fqn)
    )

    return dest_fqn

In [0]:
transform_bronze_to_silver("bronze.agent_policies")
transform_bronze_to_silver("bronze.agents")
transform_bronze_to_silver("bronze.claims")
transform_bronze_to_silver("bronze.customers")
transform_bronze_to_silver("bronze.insurance_types")
transform_bronze_to_silver("bronze.payments")
transform_bronze_to_silver("bronze.policies")

## (SQL) SHOW TABLES IN bronze

Verifica os dados gravados no formato delta lake tipo MANAGED na camada bronze.

In [0]:
%sql
SHOW TABLES IN silver

## (SQL) DESCRIBE DETAIL nome_tabela;


Vendo os detalhes de um tabela delta lake.

In [0]:
%sql
DESCRIBE DETAIL silver.policies;


## (SQL) DESCRIBE EXTENDED nome_tabela;
ou 
##(SQL) DESCRIBE TABLE EXTENDED nome_tabela;

Mostra se a tabela é MANAGED Ou EXTERNAL.


In [0]:
%sql
DESCRIBE EXTENDED silver.policies;
--DESCRIBE TABLE EXTENDED policies_bronze;